In [3]:
#Imports
import os
import re
import numpy as np
import pandas as pd
import requests
import tqdm

In [5]:
#Download Raw data from Github repo
print("Downloading data from GitHub...")

# GitHub API URL for the directory
api_url = "https://api.github.com/repos/COMP3608-GROUP-9-PROJECT/COMP3608_PROJECT/contents/RawData"

# Create local directory
os.makedirs("raw", exist_ok=True)

# Get list of files from GitHub API
response = requests.get(api_url)
if response.status_code == 200:
    files = response.json()
    for file in files:
        if file['type'] == 'file' and file['name'].endswith('.csv'): # Check if it's a file and ends with .csv
            download_url = file['download_url']
            r = requests.get(download_url)
            with open(f"raw/{file['name']}", "wb") as f:
                f.write(r.content)
            print(f"Downloaded: {file['name']}")
else:
    print(f"Failed to access API: {response.status_code}")

print("Success: All CSV files downloaded.\n")

Downloaded: Cities.csv
Downloaded: Conferences.csv
Downloaded: MConferenceTourneyGames.csv
Downloaded: MGameCities.csv
Downloaded: MMasseyOrdinals.csv
Downloaded: MNCAATourneyCompactResults.csv
Downloaded: MNCAATourneyDetailedResults.csv
Downloaded: MNCAATourneySeedRoundSlots.csv
Downloaded: MNCAATourneySeeds.csv
Downloaded: MNCAATourneySlots.csv
Downloaded: MRegularSeasonCompactResults.csv
Downloaded: MRegularSeasonDetailedResults.csv
Downloaded: MSeasons.csv
Downloaded: MSecondaryTourneyCompactResults.csv
Downloaded: MSecondaryTourneyTeams.csv
Downloaded: MTeamCoaches.csv
Downloaded: MTeamConferences.csv
Downloaded: MTeamSpellings.csv
Downloaded: MTeams.csv
Downloaded: SampleSubmissionStage1.csv
Downloaded: SampleSubmissionStage2.csv
Downloaded: SeedBenchmarkStage1.csv
Downloaded: WConferenceTourneyGames.csv
Downloaded: WGameCities.csv
Downloaded: WNCAATourneyCompactResults.csv
Downloaded: WNCAATourneyDetailedResults.csv
Downloaded: WNCAATourneySeeds.csv
Downloaded: WNCAATourneySlots

In [17]:
#Process CSV files
print("Processing local CSV files...")

# Mapping the internal variable names to their respective file pairs
data_map = {
    "seasonResults": ("MRegularSeasonDetailedResults.csv", "WRegularSeasonDetailedResults.csv"),
    "tourneyResults": ("MNCAATourneyDetailedResults.csv", "WNCAATourneyDetailedResults.csv"),
    "seeds": ("MNCAATourneySeeds.csv", "WNCAATourneySeeds.csv")
}

# Dictionary to temporarily hold our loaded dataframes
dfs = {}

for var_name, (men_file, women_file) in data_map.items():
    # Load and tag Men's data (Divison 1)
    m_df = pd.read_csv(f"raw/{men_file}").assign(Divison=1)
    # Load and tag Women's data (Divison 0)
    w_df = pd.read_csv(f"raw/{women_file}").assign(Divison=0)

    # Combine and store
    dfs[var_name] = pd.concat([m_df, w_df], ignore_index=True)

# Unpack the dictionary into specific variables
seasonResults = dfs["seasonResults"]
tourneyResults = dfs["tourneyResults"]
seeds = dfs["seeds"]

# Seeds adjustments

# tourneyResults Seasons start in 2003
seeds = seeds[seeds["Season"] >= 2003].copy()

# Convert Seed (e.g., 'W01') to integer (1)
seeds["Seed"] = seeds["Seed"].str[1:3].astype(int)


print(f"Season Results merged: {seasonResults.shape}")
print(f"Tourney Results merged: {tourneyResults.shape}")
print(f"Seed data prepared:    {seeds.shape}\n")

Processing local CSV files...
Season Results merged: (200590, 35)
Tourney Results merged: (2276, 35)
Seed data prepared:    (2896, 4)



In [8]:
seasonResults

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF,Divison
0,2003,10,1104,68,1328,62,N,0,27,58,...,16,22,10,22,8,18,9,2,20,1
1,2003,10,1272,70,1393,63,N,0,26,62,...,9,20,20,25,7,12,8,6,16,1
2,2003,11,1266,73,1437,61,N,0,24,58,...,14,23,31,22,9,12,2,5,23,1
3,2003,11,1296,56,1457,50,N,0,18,38,...,8,15,17,20,9,19,4,3,23,1
4,2003,11,1400,77,1208,71,N,0,30,61,...,17,27,21,15,12,10,7,1,14,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200585,2025,131,3471,75,3413,66,A,0,26,62,...,9,14,9,26,14,10,6,5,22,0
200586,2025,132,3192,66,3476,49,H,0,23,55,...,3,4,14,22,14,17,4,1,17,0
200587,2025,132,3250,74,3119,62,H,0,27,45,...,6,10,8,13,10,10,5,0,20,0
200588,2025,132,3293,83,3125,62,N,0,28,54,...,12,14,12,22,11,7,5,0,16,0


In [9]:
tourneyResults

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF,Divison
0,2003,134,1421,92,1411,84,N,1,32,69,...,14,31,17,28,16,15,5,0,22,1
1,2003,136,1112,80,1436,51,N,0,31,66,...,7,7,8,26,12,17,10,3,15,1
2,2003,136,1113,84,1272,71,N,0,31,59,...,14,21,20,22,11,12,2,5,18,1
3,2003,136,1141,79,1166,73,N,0,29,53,...,12,17,14,17,20,21,6,6,21,1
4,2003,136,1143,76,1301,74,N,1,27,64,...,15,20,10,26,16,14,5,8,19,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2271,2024,147,3163,80,3425,73,A,0,28,58,...,18,20,10,25,10,9,6,4,20,0
2272,2024,147,3234,94,3261,87,H,0,32,69,...,11,17,21,28,15,13,6,6,21,0
2273,2024,151,3234,71,3163,69,N,0,27,59,...,3,4,6,22,21,14,15,1,18,0
2274,2024,151,3376,78,3301,59,N,0,33,66,...,13,18,10,18,5,12,9,1,9,0


In [18]:
seeds

,Season,Seed,TeamID,Divison
1154,2003,1,1328,1
1155,2003,2,1448,1
1156,2003,3,1393,1
1157,2003,4,1257,1
1158,2003,5,1280,1
...,...,...,...,...
4365,2025,12,3193,0
4366,2025,13,3251,0
4367,2025,14,3195,0
4368,2025,15,3117,0


In [19]:
# Data preparation

print("Normalizing pace and structuring symmetric game pairings...")

def format_game_data(df):
    # Group the columns logically for better readability
    meta_cols = ["Season", "DayNum", "NumOT", "Divison"]
    win_cols = ["WTeamID", "WScore", "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA",
                "WOR", "WDR", "WAst", "WTO", "WStl", "WBlk", "WPF"]
    loss_cols = ["LTeamID", "LScore", "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA",
                 "LOR", "LDR", "LAst", "LTO", "LStl", "LBlk", "LPF"]

    # Isolate the required dataframe
    df = df[meta_cols + win_cols + loss_cols].copy()

    # Adjustments

    # Normalize stats to a standard 40-minute regulation game
    # Overtime is 5 mins. Pace Factor = (40 + 5 * OT) / 40
    pace_factor = 1 + (df["NumOT"] * 5 / 40)

    # Dynamically grab all columns that are game stats (excluding metadata and Team IDs)
    stat_cols = [c for c in df.columns if c not in meta_cols + ["WTeamID", "LTeamID"]]

    # Apply pace adjustment
    for col in stat_cols:
        df[col] = df[col] / pace_factor

    # Create symmetric Team 1 (T1) vs Team 2 (T2) pairings
    # Perspective 1: T1 is the Winner, T2 is the Loser
    df_p1 = df.copy()
    df_p1.columns = [c.replace("W", "T1").replace("L", "T2") for c in df.columns]

    # Perspective 2: T1 is the Loser, T2 is the Winner
    df_p2 = df.copy()
    df_p2.columns = [c.replace("L", "T1").replace("W", "T2") for c in df.columns]

    # Stack them on top of each other
    symmetric_df = pd.concat([df_p1, df_p2], ignore_index=True)

    # 3. Target Variable and Context Features
    symmetric_df["PointDifference"] = symmetric_df["T1Score"] - symmetric_df["T2Score"]
    symmetric_df["Win"] = (symmetric_df["PointDifference"] > 0).astype(int)

    # Vectorized string check to identify Division (Men's IDs start with '1', Women's with '3')
    symmetric_df["Divison"] = symmetric_df["T1TeamID"].astype(str).str.startswith("1").astype(int)

    return symmetric_df

# Apply the function to our datasets
seasonData = format_game_data(seasonResults)
tourneyData = format_game_data(tourneyResults)

print(f"Symmetric Season Data:  {seasonData.shape}")
print(f"Symmetric Tourney Data: {tourneyData.shape}\n")

Normalizing pace and structuring symmetric game pairings...
Symmetric Season Data:  (401180, 36)
Symmetric Tourney Data: (4552, 36)



In [20]:
seasonData

,Season,DayNum,NumOT,Divison,T1TeamID,T1Score,T1FGM,T1FGA,T1FGM3,T1FGA3,...,T2FTA,T2OR,T2DR,T2Ast,T2TO,T2Stl,T2Blk,T2PF,PointDifference,Win
0,2003,10,0,1,1104,68.0,27.0,58.0,3.0,14.0,...,22.0,10.0,22.0,8.0,18.0,9.0,2.0,20.0,6.0,1
1,2003,10,0,1,1272,70.0,26.0,62.0,8.0,20.0,...,20.0,20.0,25.0,7.0,12.0,8.0,6.0,16.0,7.0,1
2,2003,11,0,1,1266,73.0,24.0,58.0,8.0,18.0,...,23.0,31.0,22.0,9.0,12.0,2.0,5.0,23.0,12.0,1
3,2003,11,0,1,1296,56.0,18.0,38.0,3.0,9.0,...,15.0,17.0,20.0,9.0,19.0,4.0,3.0,23.0,6.0,1
4,2003,11,0,1,1400,77.0,30.0,61.0,6.0,14.0,...,27.0,21.0,15.0,12.0,10.0,7.0,1.0,14.0,6.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401175,2025,131,0,0,3413,66.0,24.0,67.0,9.0,29.0,...,28.0,8.0,31.0,10.0,11.0,6.0,1.0,20.0,-9.0,0
401176,2025,132,0,0,3476,49.0,21.0,57.0,4.0,20.0,...,18.0,10.0,22.0,11.0,9.0,8.0,1.0,8.0,-17.0,0
401177,2025,132,0,0,3119,62.0,25.0,56.0,6.0,17.0,...,17.0,5.0,25.0,15.0,15.0,6.0,0.0,12.0,-12.0,0
401178,2025,132,0,0,3125,62.0,24.0,68.0,2.0,21.0,...,15.0,5.0,33.0,21.0,13.0,2.0,3.0,15.0,-21.0,0


In [21]:
tourneyData

,Season,DayNum,NumOT,Divison,T1TeamID,T1Score,T1FGM,T1FGA,T1FGM3,T1FGA3,...,T2FTA,T2OR,T2DR,T2Ast,T2TO,T2Stl,T2Blk,T2PF,PointDifference,Win
0,2003,134,1,1,1421,81.777778,28.444444,61.333333,9.777778,25.777778,...,27.555556,15.111111,24.888889,14.222222,13.333333,4.444444,0.000000,19.555556,7.111111,1
1,2003,136,0,1,1112,80.000000,31.000000,66.000000,7.000000,23.000000,...,7.000000,8.000000,26.000000,12.000000,17.000000,10.000000,3.000000,15.000000,29.000000,1
2,2003,136,0,1,1113,84.000000,31.000000,59.000000,6.000000,14.000000,...,21.000000,20.000000,22.000000,11.000000,12.000000,2.000000,5.000000,18.000000,13.000000,1
3,2003,136,0,1,1141,79.000000,29.000000,53.000000,3.000000,7.000000,...,17.000000,14.000000,17.000000,20.000000,21.000000,6.000000,6.000000,21.000000,6.000000,1
4,2003,136,1,1,1143,67.555556,24.000000,56.888889,6.222222,17.777778,...,17.777778,8.888889,23.111111,14.222222,12.444444,4.444444,7.111111,16.888889,1.777778,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4547,2024,147,0,0,3425,73.000000,23.000000,70.000000,9.000000,29.000000,...,27.000000,5.000000,30.000000,17.000000,12.000000,6.000000,5.000000,21.000000,-7.000000,0
4548,2024,147,0,0,3261,87.000000,34.000000,88.000000,8.000000,24.000000,...,22.000000,3.000000,29.000000,16.000000,11.000000,6.000000,3.000000,15.000000,-7.000000,0
4549,2024,151,0,0,3163,69.000000,29.000000,63.000000,8.000000,25.000000,...,14.000000,9.000000,23.000000,12.000000,16.000000,7.000000,1.000000,9.000000,-2.000000,0
4550,2024,151,0,0,3301,59.000000,20.000000,62.000000,6.000000,23.000000,...,4.000000,10.000000,34.000000,18.000000,15.000000,10.000000,6.000000,16.000000,-19.000000,0
